# 01. Data setup and loading

This notebook:
1. Inspects the 8 raw dunnhumby CSV files: structure, types, nulls, duplicates, and anomalous values, before assuming anything about them
2. Documents and applies two targeted cleaning fixes where the raw data didn't hold up (whitespace-only values in `product.csv`, exact duplicate rows in `coupon.csv`)
3. Designs and creates the 8-table PostgreSQL schema, with primary/foreign keys grounded in what Sections 3–5 actually found, not assumed
4. Loads all 8 tables and verifies the load inside PostgreSQL (row counts, `NULL` counts, and quantity and sales totals against the source files)
5. Verifies the candidate primary key of `causal_data` in SQL, adds it after loading, and documents the resulting relational structure directly from PostgreSQL

Business transformation (RFM, clustering, cohorts, campaign analysis) is out of scope here and happens in the notebooks that follow, each scoped to its own question.

**Prerequisites:** a `dunnhumby` database must already exist in your local PostgreSQL server (created via pgAdmin), and a `.env` file with the connection variables (`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`) must be present in the project root. The `.env` file is excluded from version control via `.gitignore`; `.env.example` shows the expected format. The notebook drops and recreates its own 8 tables (Section 6); any other tables in that database are not touched. The full run loads 36.8M rows into `causal_data`, so the loading and key-creation steps (Sections 7 and 8) can take several minutes.

## 1. Environment setup

We load configuration from environment variables (`.env`) rather than hardcoding credentials in the notebook. This keeps secrets out of version control and makes the notebook portable across machines. The cell also fails fast, with an explicit message, if a variable is missing or the raw data folder is not where the notebook expects it, instead of failing later with a confusing connection or file error.

In [1]:
import io
import math
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import bindparam, create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

RAW_DATA_DIR = Path("../data/raw/dunnhumby")

required = {
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise EnvironmentError(f"Missing variables in .env: {missing}")
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data folder not found: {RAW_DATA_DIR.resolve()}")

## 2. Database connection

Before loading any data, we validate that the connection to PostgreSQL works.
If this fails, everything downstream would fail silently or with confusing errors,
so we check it explicitly first. The connection URL is built with SQLAlchemy's `URL.create`, which escapes special characters in the password.

In [2]:
engine = create_engine(
    URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME,
    )
)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).fetchone()
    print(version)

('PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit',)


The connection succeeded, confirming the local server is running PostgreSQL 18.3. Credentials and network configuration are correctly resolved from `.env`.

With connectivity validated, we move on to inspecting the raw CSV files before touching PostgreSQL at all: the database schema should be derived from evidence, not assumed.

## 3. Raw file inspection

Before defining any PostgreSQL schema, we inspect the actual CSV files:
column names, inferred dtypes, row counts, and a sample of rows. This avoids
assuming a schema from memory or documentation that may not match the real files.

In [3]:
csv_files = [
    "campaign_desc.csv",
    "campaign_table.csv",
    "causal_data.csv",
    "coupon.csv",
    "coupon_redempt.csv",
    "hh_demographic.csv",
    "product.csv",
    "transaction_data.csv",
]

file_summaries = {}

for filename in csv_files:
    filepath = RAW_DATA_DIR / filename
    df_sample = pd.read_csv(filepath, nrows=5)
    with open(filepath, "rb") as f:
        n_rows = sum(1 for _ in f) - 1  # subtract header
    size_bytes = filepath.stat().st_size

    file_summaries[filename] = {
        "columns": list(df_sample.columns),
        "dtypes": df_sample.dtypes.to_dict(),
        "n_rows": n_rows,
        "size_mb": size_bytes / (1024 ** 2),
        "size_kb": size_bytes / 1024,
    }

    print(f"--- {filename} ---")
    print(f"Rows: {n_rows}")
    print(f"Size: {size_bytes / (1024 ** 2):,.1f} MB")
    print(f"Columns: {list(df_sample.columns)}")
    print(df_sample.head())
    print()

--- campaign_desc.csv ---
Rows: 30
Size: 0.0 MB
Columns: ['DESCRIPTION', 'CAMPAIGN', 'START_DAY', 'END_DAY']
  DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
0       TypeB        24        659      719
1       TypeC        15        547      708
2       TypeB        25        659      691
3       TypeC        20        615      685
4       TypeB        23        646      684

--- campaign_table.csv ---
Rows: 7208
Size: 0.1 MB
Columns: ['DESCRIPTION', 'household_key', 'CAMPAIGN']
  DESCRIPTION  household_key  CAMPAIGN
0       TypeA             17        26
1       TypeA             27        26
2       TypeA            212        26
3       TypeA            208        26
4       TypeA            192        26

--- causal_data.csv ---
Rows: 36786524
Size: 663.6 MB
Columns: ['PRODUCT_ID', 'STORE_ID', 'WEEK_NO', 'display', 'mailer']
   PRODUCT_ID  STORE_ID  WEEK_NO  display mailer
0       26190       286       70        0      A
1       26190       288       70        0      A
2       26190    

All 8 files loaded and displayed correctly, confirming column names and file names match what we expect. Two things stand out and are worth flagging before designing the schema:

- `product.csv`: `CURR_SIZE_OF_PRODUCT` shows blank values in the sample (rows 1 and 2). We still need to confirm whether pandas reads these as `NaN` or as empty strings, since that changes how the column should be defined in PostgreSQL.
- The dtypes captured here come from a 5-row sample (`nrows=5`), which is not reliable for columns that might contain nulls further down the file: a `NaN` appearing later would silently turn an `int64` column into `float64` when the full file is read. The schema is not finalized on this sample; data types are confirmed once we have checked the full files (Section 3.2).

Next, we confirm what the blank values in `product.csv` actually are (Section 3.1), repeat the null and dtype checks on the full files (Section 3.2), consolidate row counts and file sizes (Section 3.3), and check the temporal ranges (Section 3.4).

### 3.1 Confirming blank values in product.csv

The 5-row sample showed blank entries in `CURR_SIZE_OF_PRODUCT`. We check whether pandas is reading these as `NaN` (missing) or as literal empty strings (`""`), since the two require different handling in PostgreSQL: `NaN` maps to `NULL`, an empty string is a valid (if uninformative) value. Note that pandas converts truly empty fields to `NaN` on read by default, so the empty-string count is expected to be zero; it is reported anyway to rule that case out explicitly.

In [4]:
product_full = pd.read_csv(RAW_DATA_DIR / "product.csv")

n_nan = product_full["CURR_SIZE_OF_PRODUCT"].isna().sum()
n_empty_string = (product_full["CURR_SIZE_OF_PRODUCT"] == "").sum()
n_total = len(product_full)

print(f"Total rows: {n_total}")
print(f"NaN values: {n_nan} ({n_nan / n_total:.1%})")
print(f"Empty string values: {n_empty_string} ({n_empty_string / n_total:.1%})")

Total rows: 92353
NaN values: 0 (0.0%)
Empty string values: 0 (0.0%)


The full-file check found zero `NaN` values and zero empty strings, yet the sample clearly showed blank-looking cells. The most likely explanation is that these fields contain whitespace characters (e.g. a single space) rather than being truly empty: pandas does not treat whitespace as missing, and a whitespace string fails an `== ""` comparison, so neither `.isna()` nor `== ""` can catch it.

This matters for the schema: if loaded as-is, those rows would land in PostgreSQL as a non-null but meaningless value instead of a clean `NULL`. Before deciding how to define this column, we check explicitly for whitespace-only strings.

In [5]:
n_whitespace_only = (
    product_full["CURR_SIZE_OF_PRODUCT"]
    .str.strip()
    .eq("")
    .sum()
)

print(f"Whitespace-only values: {n_whitespace_only} ({n_whitespace_only / n_total:.1%})")

Whitespace-only values: 30607 (33.1%)


Confirmed: 30,607 of 92,353 rows (33.1%) in `CURR_SIZE_OF_PRODUCT` are whitespace-only
strings, not true `NaN`. This is not a marginal edge case: it's a third of the product
catalog, so the transformation is stated and quantified explicitly rather than applied silently (the same rule is applied to the `coupon.csv` duplicates in Section 5.2).

The decision is to normalize these whitespace-only values to real `NULL` before loading
`product` into PostgreSQL. A space character carries no product-size information, so
treating it as a proper null is more accurate than leaving it as an opaque non-empty
string, and it lets PostgreSQL enforce and query nullability correctly on this column.

Before normalizing only this column, we check every text column of the six small tables for the same problem, rather than assuming `CURR_SIZE_OF_PRODUCT` is the only one affected because it happened to show up in the 5-row sample.

In [6]:
small_tables = [f for f in csv_files if f not in ("causal_data.csv", "transaction_data.csv")]

whitespace_only = {}
for filename in small_tables:
    df_scan = pd.read_csv(RAW_DATA_DIR / filename)
    for column in df_scan.select_dtypes(include=["object", "string"]).columns:
        n_blank = int(df_scan[column].str.strip().eq("").sum())
        if n_blank:
            whitespace_only[(filename, column)] = n_blank

for (filename, column), n_blank in sorted(whitespace_only.items()):
    print(f"{filename} — {column}: {n_blank:,} whitespace-only values")

product.csv — COMMODITY_DESC: 15 whitespace-only values
product.csv — CURR_SIZE_OF_PRODUCT: 30,607 whitespace-only values
product.csv — DEPARTMENT: 15 whitespace-only values
product.csv — SUB_COMMODITY_DESC: 15 whitespace-only values


`CURR_SIZE_OF_PRODUCT` is not the only affected column: a small number of whitespace-only values also appear in other text columns of `product.csv`. No other small table is affected. Before normalizing, we check whether the smaller cases are concentrated in the same rows (a handful of incomplete product records) or scattered across different ones.

In [7]:
other_product_columns = sorted(
    column for (filename, column) in whitespace_only
    if filename == "product.csv" and column != "CURR_SIZE_OF_PRODUCT"
)

if other_product_columns:
    blank_masks = {
        column: product_full[column].str.strip().eq("")
        for column in other_product_columns
    }
    overlap = pd.DataFrame(blank_masks)
    print(overlap.sum().rename("n_blank"))
    print(f"\nRows blank in ALL of {other_product_columns}: {overlap.all(axis=1).sum()}")
    print(f"Rows blank in AT LEAST ONE of them: {overlap.any(axis=1).sum()}")
else:
    print("No other column of product.csv is affected.")

COMMODITY_DESC        15
DEPARTMENT            15
SUB_COMMODITY_DESC    15
Name: n_blank, dtype: int64

Rows blank in ALL of ['COMMODITY_DESC', 'DEPARTMENT', 'SUB_COMMODITY_DESC']: 15
Rows blank in AT LEAST ONE of them: 15


We now normalize every affected column of `product.csv` the same way: whitespace-only values become `NULL`, and the number of values converted is reported per column, so the correction is fully quantified regardless of how many columns turned out to need it.

In [8]:
affected_columns = sorted(
    column for (filename, column) in whitespace_only if filename == "product.csv"
)

normalized_counts = {}
for column in affected_columns:
    stripped = product_full[column].str.strip()
    product_full[column] = stripped.replace("", pd.NA)
    normalized_counts[column] = int(product_full[column].isna().sum())

for column, n_null in normalized_counts.items():
    print(f"{column}: {n_null:,} nulls after normalization ({n_null / n_total:.1%})")

COMMODITY_DESC: 15 nulls after normalization (0.0%)
CURR_SIZE_OF_PRODUCT: 30,607 nulls after normalization (33.1%)
DEPARTMENT: 15 nulls after normalization (0.0%)
SUB_COMMODITY_DESC: 15 nulls after normalization (0.0%)


`.str.strip()` also trims leading and trailing spaces on non-blank values in every normalized column, without changing their content. `product_full` is now ready to load with proper `NULL` semantics in all four affected columns. The raw CSV is left untouched; the correction is applied only in memory before loading.

The real invariant this section needs is not "only `CURR_SIZE_OF_PRODUCT` has this problem" (an assumption from a 5-row sample), but "no whitespace-only value survives normalization, in any small table." We check that directly.

In [9]:
remaining = {}
for filename in small_tables:
    df_check = product_full if filename == "product.csv" else pd.read_csv(RAW_DATA_DIR / filename)
    for column in df_check.select_dtypes(include=["object", "string"]).columns:
        n_blank = int(df_check[column].astype("string").str.strip().eq("").sum())
        if n_blank:
            remaining[(filename, column)] = n_blank

assert not remaining, f"Whitespace-only values remain after normalization: {remaining}"
print("No whitespace-only values remain in any small table after normalization.")

No whitespace-only values remain in any small table after normalization.


`transaction_data` has no text columns, and the codes stored in the `display` and `mailer` columns of `causal_data` are inventoried after loading (Section 7.2), so this check covers every column that could carry the issue.

This closes Section 3.1. Sections 3.2–3.4 complete the inspection: full-file dtypes and nulls, scale, and temporal ranges.

### 3.2 Full-file null and dtype check

We repeat the inspection from Section 3, but this time reading each file in
full rather than a 5-row sample, and compare inferred dtypes against what the
sample suggested. For the two large tables (`causal_data`, `transaction_data`),
we scan for nulls using chunked reads instead of loading the entire file into
memory at once.

In [10]:
small_tables = [f for f in csv_files if f not in ("causal_data.csv", "transaction_data.csv")]

for filename in small_tables:
    df_full = pd.read_csv(RAW_DATA_DIR / filename)
    sample_dtypes = file_summaries[filename]["dtypes"]
    full_dtypes = df_full.dtypes.to_dict()

    mismatches = {
        col: (str(sample_dtypes.get(col)), str(full_dtypes[col]))
        for col in full_dtypes
        if str(sample_dtypes.get(col)) != str(full_dtypes[col])
    }
    null_counts = df_full.isna().sum()
    null_counts = null_counts[null_counts > 0]

    print(f"--- {filename} ---")
    print(f"Columns with nulls: {dict(null_counts) if len(null_counts) else 'none'}")
    print(f"Dtype mismatches vs 5-row sample: {mismatches if mismatches else 'none'}")
    print()

large_tables = ["causal_data.csv", "transaction_data.csv"]
chunk_size = 500_000

for filename in large_tables:
    filepath = RAW_DATA_DIR / filename
    null_totals = None
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        chunk_nulls = chunk.isna().sum()
        null_totals = chunk_nulls if null_totals is None else null_totals + chunk_nulls

    null_totals = null_totals[null_totals > 0]
    print(f"--- {filename} (scanned in chunks of {chunk_size:,}) ---")
    print(f"Columns with nulls: {dict(null_totals) if len(null_totals) else 'none'}")
    print()

--- campaign_desc.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: none

--- campaign_table.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: none

--- coupon.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: none

--- coupon_redempt.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: none

--- hh_demographic.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: {'HOUSEHOLD_SIZE_DESC': ('int64', 'object')}

--- product.csv ---
Columns with nulls: none
Dtype mismatches vs 5-row sample: none

--- causal_data.csv (scanned in chunks of 500,000) ---
Columns with nulls: none

--- transaction_data.csv (scanned in chunks of 500,000) ---
Columns with nulls: none



**Nulls:** no nulls in any of the 8 files, including the two large tables scanned in chunks. **Dtypes:** the comparison against the 5-row sample was run only on the six small tables; five show no mismatch. `hh_demographic` is the exception: `HOUSEHOLD_SIZE_DESC` was inferred as `int64` from the sample but is `object` (text) in the full 801 rows, which indicates at least one non-numeric label: `value_counts()` on the full file (next cell) shows the value is "5+". For `causal_data` and `transaction_data`, dtypes were not compared here; the mixed-type `display` column of `causal_data` is handled with an explicit dtype when the table is loaded (Section 7.1).

This is precisely the scenario the sample-based dtypes could not be trusted for. `HOUSEHOLD_SIZE_DESC` will be defined as `VARCHAR` in PostgreSQL, not `INTEGER`, the opposite of what the initial sample would have suggested. With this confirmed for the six small tables, we move on to consolidating scale (row counts, size) before designing the schema.

In [11]:
print(pd.read_csv(RAW_DATA_DIR / "hh_demographic.csv")["HOUSEHOLD_SIZE_DESC"].value_counts())

HOUSEHOLD_SIZE_DESC
2     318
1     255
3     109
5+     66
4      53
Name: count, dtype: int64


66 of 801 rows (8.2%) hold the literal string `"5+"`; the other four categories (`1`–`4`) are numeric-looking but were already read as `object` due to that mixed column, confirming `VARCHAR` is the correct type for `household_size_desc`.

### 3.3 Summary table

A consolidated view of the 8 raw files: row count, file size, and number of
columns. This gives a quick sense of scale before moving into PostgreSQL,
in particular, which tables will require chunked loading.

In [12]:
summary_df = pd.DataFrame([
    {
        "table": filename.replace(".csv", ""),
        "n_rows": data["n_rows"],
        "n_columns": len(data["columns"]),
        "size_mb": round(data["size_mb"], 1),
        "size_kb": round(data["size_kb"]),
    }
    for filename, data in file_summaries.items()
]).sort_values("size_kb", ascending=False).reset_index(drop=True)

summary_df

,table,n_rows,n_columns,size_mb,size_kb
0,causal_data,36786524,5,663.6,679549
1,transaction_data,2595732,12,135.2,138420
2,product,92353,7,6.1,6279
3,coupon,124548,3,2.7,2757
4,campaign_table,7208,3,0.1,94
5,coupon_redempt,2318,4,0.1,53
6,hh_demographic,801,8,0.0,43
7,campaign_desc,30,4,0.0,1


`causal_data` (36.8M rows, 663.6 MB) and `transaction_data` (2.6M rows, 135.2 MB) are, by a wide margin, the two largest tables. Everything else is under 10 MB.

Loading them with a single `df.to_sql()` call would require holding the full file in memory as a DataFrame and inserting it row by row. That is a risky and slow approach, given `causal_data`'s size in particular.

Both tables will therefore be read in chunks (`pd.read_csv(..., chunksize=...)`) and loaded with PostgreSQL's `COPY` (Section 7.1). The remaining six tables are small enough to load directly with `to_sql()`.

### 3.4 Temporal range check

`DAY` is dunnhumby's sequential time unit, not a calendar date. Before relying
on it for recency, cohorts, or campaign windows in later notebooks, we confirm
its actual range across every table that uses it, and check whether they cover
the same window rather than assuming it.
The columns of `transaction_data` needed here and in Sections 4, 5 and 5.3 are read once (next cell) and reused, instead of re-reading a 135 MB file in every section.

In [13]:
trans_core = pd.read_csv(
    RAW_DATA_DIR / "transaction_data.csv",
    usecols=["household_key", "BASKET_ID", "PRODUCT_ID", "DAY", "WEEK_NO", "SALES_VALUE", "QUANTITY"],
)
print(f"trans_core: {len(trans_core):,} rows, {trans_core.shape[1]} columns")

trans_core: 2,595,732 rows, 7 columns


In [14]:
print(f"transaction_data.csv — DAY range: {trans_core['DAY'].min()} to {trans_core['DAY'].max()}")

redempt_days = pd.read_csv(RAW_DATA_DIR / "coupon_redempt.csv", usecols=["DAY"])["DAY"]
print(f"coupon_redempt.csv — DAY range: {redempt_days.min()} to {redempt_days.max()}")

causal_weeks = set()
for chunk in pd.read_csv(RAW_DATA_DIR / "causal_data.csv", usecols=["WEEK_NO"], chunksize=2_000_000):
    causal_weeks.update(int(w) for w in chunk["WEEK_NO"].unique())

causal_min, causal_max = min(causal_weeks), max(causal_weeks)
print(f"causal_data.csv — WEEK_NO range: {causal_min} to {causal_max}")

campaign_range = pd.read_csv(RAW_DATA_DIR / "campaign_desc.csv", usecols=["CAMPAIGN", "START_DAY", "END_DAY"])
print(f"campaign_desc.csv — START_DAY range: {campaign_range['START_DAY'].min()} to {campaign_range['START_DAY'].max()}")
print(f"campaign_desc.csv — END_DAY range: {campaign_range['END_DAY'].min()} to {campaign_range['END_DAY'].max()}")

last_transaction_day = trans_core["DAY"].max()
late_campaigns = campaign_range[campaign_range["END_DAY"] > last_transaction_day]
print(f"\nCampaigns ending after the last transaction day ({last_transaction_day}): {len(late_campaigns)}")
print(late_campaigns.to_string(index=False))

transaction_data.csv — DAY range: 1 to 711
coupon_redempt.csv — DAY range: 225 to 704
causal_data.csv — WEEK_NO range: 9 to 101
campaign_desc.csv — START_DAY range: 224 to 659
campaign_desc.csv — END_DAY range: 264 to 719

Campaigns ending after the last transaction day (711): 1
 CAMPAIGN  START_DAY  END_DAY
       24        659      719


In [15]:
trans_weeks = {int(w) for w in trans_core["WEEK_NO"].unique()}

weeks_without_causal = sorted(trans_weeks - causal_weeks)
internal_gaps = sorted(set(range(causal_min, causal_max + 1)) - causal_weeks)

print(f"transaction_data.csv — WEEK_NO range: {min(trans_weeks)} to {max(trans_weeks)}")
print(f"causal_data.csv      — WEEK_NO range: {causal_min} to {causal_max} ({len(causal_weeks)} distinct weeks)")
print(f"Transaction weeks with no causal_data coverage: {weeks_without_causal}")
print(f"Gaps inside causal_data's own week range: {internal_gaps if internal_gaps else 'none'}")

week_span = trans_core.groupby("WEEK_NO")["DAY"].agg(first_day="min", last_day="max", n_days="nunique")
last_week = week_span.index[-1]
last = week_span.iloc[-1]
kind = "partial" if last["n_days"] < 7 else "complete"
print(
    f"Final transaction week ({last_week}) is a {kind} week: "
    f"DAY {last['first_day']} to {last['last_day']} ({last['n_days']} distinct days with transactions)"
)

# Sanity checks against the coverage documented below
assert not internal_gaps
assert weeks_without_causal == list(range(1, 9)) + [102]

transaction_data.csv — WEEK_NO range: 1 to 102
causal_data.csv      — WEEK_NO range: 9 to 101 (93 distinct weeks)
Transaction weeks with no causal_data coverage: [1, 2, 3, 4, 5, 6, 7, 8, 102]
Gaps inside causal_data's own week range: none
Final transaction week (102) is a partial week: DAY 706 to 711 (6 distinct days with transactions)


`transaction_data` covers the full range (`DAY` 1 to 711), and `campaign_desc`'s earliest `START_DAY` (224) falls within it, one day before the first redemption in `coupon_redempt` (`DAY` 225): redemptions can't happen before a campaign exists.

Two discrepancies are worth flagging rather than ignoring. First, `causal_data` covers `WEEK_NO` 9 to 101, while `transaction_data` covers `WEEK_NO` 1 to 102. The comparison uses the weeks actually present in each table instead of assuming the range is contiguous, and it also checks for gaps inside `causal_data`'s own range (there are none). Weeks 1–8 and week 102 therefore have no corresponding promotional (display/mailer) data at all: the first 8 weeks of the observation window and its final week. The last printed line reports how many distinct days of transactions the final week contains; a partial week must not be treated like a full one in any weekly aggregation. Any future analysis joining transactions with causal data should account for both gaps, rather than assuming causal coverage is complete.

Second, `campaign_desc`'s `END_DAY` reaches 719, eight days past `transaction_data`'s maximum of 711; the cell lists the campaigns recorded as ending after the transaction window closes. This is a minor edge case, not a data quality problem, but worth knowing before computing any campaign-window metrics in `05_campaigns_and_coupons.ipynb`.

## 4. Household coverage check

We check how many distinct households appear in `transaction_data` (households
that actually made purchases) versus how many have a corresponding row in
`hh_demographic` (households with demographic info available). This directly
affects which future analyses can rely on demographic segmentation.

In [16]:
demo_hh = pd.read_csv(RAW_DATA_DIR / "hh_demographic.csv", usecols=["household_key"])

trans_hh_unique = set(trans_core["household_key"].unique())
demo_hh_unique = set(demo_hh["household_key"].unique())

n_trans = len(trans_hh_unique)
n_demo = len(demo_hh_unique)
n_overlap = len(trans_hh_unique & demo_hh_unique)

print(f"Distinct households in transaction_data: {n_trans}")
print(f"Distinct households in hh_demographic:   {n_demo}")
print(f"Households with both transactions AND demographic data: {n_overlap}")
print(f"Coverage: {n_overlap / n_trans:.1%} of purchasing households have demographic data")

Distinct households in transaction_data: 2500
Distinct households in hh_demographic:   801
Households with both transactions AND demographic data: 801
Coverage: 32.0% of purchasing households have demographic data


2,500 households appear in `transaction_data`, but only 801 have a row in `hh_demographic`: 32.0% coverage. All 801 of those households also appear in `transaction_data` (801 overlap out of 801 demographic households), so `hh_demographic` is a proper subset of the purchasing households: 1,699 of 2,500 households have no demographic record. This notebook does not establish why, so it is documented as a coverage limitation of the source, not as a loading error or a sampling design.

We do not fill or infer missing demographics, and no household is dropped for lacking one. Any analysis that uses demographic attributes will be explicitly scoped to these 801 households (32.0%). Note that `hh_demographic` also contains coded unknowns (e.g. `Unknown`, `None/Unknown`, `U`); they are preserved as-is, not imputed.

## 5. Null and duplicate check on key columns

We check the full files (not just the 5-row sample) for nulls and duplicates
in the columns that will act as primary/foreign keys. This is critical before
designing the PostgreSQL schema: a key column with nulls or duplicates changes
how (or whether) we can enforce constraints.

In [17]:
key_checks = {
    "transaction_data.csv": ["household_key", "BASKET_ID", "PRODUCT_ID"],
    "hh_demographic.csv": ["household_key"],
    "product.csv": ["PRODUCT_ID"],
    "campaign_desc.csv": ["CAMPAIGN"],
    "campaign_table.csv": ["household_key", "CAMPAIGN"],
    "coupon.csv": ["COUPON_UPC", "PRODUCT_ID"],
    "coupon_redempt.csv": ["household_key", "COUPON_UPC", "CAMPAIGN"],
}

for filename, cols in key_checks.items():
    if filename == "transaction_data.csv":
        df = trans_core[cols]
    else:
        df = pd.read_csv(RAW_DATA_DIR / filename, usecols=cols)
    print(f"--- {filename} ---")
    print("Nulls per column:")
    print(df.isna().sum())
    print(f"Repeated key combinations across {cols} (rows beyond the first occurrence): {df.duplicated(subset=cols).sum()}")
    print()

--- transaction_data.csv ---
Nulls per column:
household_key    0
BASKET_ID        0
PRODUCT_ID       0
dtype: int64
Repeated key combinations across ['household_key', 'BASKET_ID', 'PRODUCT_ID'] (rows beyond the first occurrence): 0

--- hh_demographic.csv ---
Nulls per column:
household_key    0
dtype: int64
Repeated key combinations across ['household_key'] (rows beyond the first occurrence): 0

--- product.csv ---
Nulls per column:
PRODUCT_ID    0
dtype: int64
Repeated key combinations across ['PRODUCT_ID'] (rows beyond the first occurrence): 0

--- campaign_desc.csv ---
Nulls per column:
CAMPAIGN    0
dtype: int64
Repeated key combinations across ['CAMPAIGN'] (rows beyond the first occurrence): 0

--- campaign_table.csv ---
Nulls per column:
household_key    0
CAMPAIGN         0
dtype: int64
Repeated key combinations across ['household_key', 'CAMPAIGN'] (rows beyond the first occurrence): 0

--- coupon.csv ---
Nulls per column:
COUPON_UPC    0
PRODUCT_ID    0
dtype: int64
Repeated 

Zero nulls across the key columns checked in the seven files covered by this cell (`causal_data` is checked separately in Sections 3.2 and 8.1).

The absence of nulls means we can safely define `NOT NULL` constraints on these columns in PostgreSQL. The duplicates can't be interpreted yet: both checks excluded a column (`CAMPAIGN` and `DAY`, respectively) that could fully explain the repetition as a legitimate business event rather than a data error.

Before deciding on primary keys or deduplication, we re-run both duplicate checks using every column in the file, to separate genuine duplicate rows from valid repeated combinations.

### 5.1 Disambiguating duplicates in coupon.csv and coupon_redempt.csv

The previous check excluded `CAMPAIGN` (coupon.csv) and `DAY` (coupon_redempt.csv)
from the duplicate subset. We re-check using *all* columns to distinguish
genuine duplicate rows from legitimate repeated combinations across different
campaigns or days.

In [18]:
coupon_full = pd.read_csv(RAW_DATA_DIR / "coupon.csv")
coupon_redempt_full = pd.read_csv(RAW_DATA_DIR / "coupon_redempt.csv")

n_dup_coupon_all_cols = coupon_full.duplicated().sum()
n_dup_redempt_all_cols = coupon_redempt_full.duplicated().sum()

print(f"coupon.csv — fully duplicated rows (ALL columns): {n_dup_coupon_all_cols}")
print(f"coupon_redempt.csv — fully duplicated rows (ALL columns): {n_dup_redempt_all_cols}")

coupon.csv — fully duplicated rows (ALL columns): 5164
coupon_redempt.csv — fully duplicated rows (ALL columns): 0


`coupon_redempt.csv` has 0 fully duplicated rows, the 30 flagged earlier are explained entirely by different `DAY` values (the same household redeeming the same coupon on separate days). `coupon.csv`, however, still shows 5,164 fully duplicated rows even using all 3 of its columns.

For `coupon_redempt.csv`, this confirms the repeats are genuine business events, not noise: its primary key is (`household_key`, `COUPON_UPC`, `CAMPAIGN`, `DAY`), no cleaning needed. For `coupon.csv`, since all 3 columns are identical between duplicate rows and there is no further column left to explain the repetition, these are redundant rows carrying no additional information, most likely an artifact of how the source file was generated.

`coupon_redempt.csv` is used as-is. `coupon.csv` needs its exact duplicates removed and documented before loading, following the same principle as every correction in this notebook: no silent drops, and the proportion removed is stated explicitly.

### 5.2 Handling exact duplicates in coupon.csv

`coupon.csv` contains 5,164 fully duplicate rows (identical `COUPON_UPC`,
`PRODUCT_ID`, and `CAMPAIGN`). Since these three columns are the entire row,
an exact duplicate carries no additional information, so it is treated as
redundant, not as a legitimate repeated business event (unlike
`coupon_redempt.csv`, where repeats are explained by a `DAY` column we
excluded from that check and turned out to be genuine).

We document the proportion removed and keep the deduplication explicit,
rather than silently dropping rows inside a later loading step.

In [19]:
n_before = len(coupon_full)
pct_duplicated = n_dup_coupon_all_cols / n_before

print(f"Total rows before dedup: {n_before}")
print(f"Exact duplicate rows: {n_dup_coupon_all_cols} ({pct_duplicated:.1%})")

coupon_clean = coupon_full.drop_duplicates().reset_index(drop=True)

print(f"Total rows after dedup: {len(coupon_clean)}")

Total rows before dedup: 124548
Exact duplicate rows: 5164 (4.1%)
Total rows after dedup: 119384


5,164 of 124,548 rows (4.1%) in `coupon.csv` were exact duplicates and have been removed, leaving 119,384 rows in `coupon_clean`.

`coupon_clean`, not the original `coupon_full`, is what gets loaded into PostgreSQL. Its primary key is (`COUPON_UPC`, `PRODUCT_ID`, `CAMPAIGN`).

This closes Section 5.2. Sections 5.1–5.2 gave us what we need to define primary keys with confidence for the tables checked here: no nulls in the key columns, one legitimate composite key discovered (`coupon_redempt`), and one genuine data-quality issue found, quantified, and resolved (`coupon.csv`). Section 5.3 completes the inspection with an anomalous-value check. After that, the notebook moves from *inspection* to *construction*: defining the 8 `CREATE TABLE` statements in PostgreSQL based on this evidence, then loading each table (`to_sql()` for the six small tables, chunked `COPY` for `transaction_data` and `causal_data`).

### 5.3 Anomalous value check

Beyond nulls and duplicates, we check for values that are structurally valid
(correct type, not missing) but don't make business sense: negative sales or
quantities in transactions, and campaigns where the end date precedes the
start date. This is detection and documentation only, the treatment decision
for non-positive `sales_value` was already scoped to `02_rfm.ipynb` as part of the
RFM design, since it directly affects the Monetary component.

In [20]:
n_total_trans = len(trans_core)

sales_zero = int((trans_core["SALES_VALUE"] == 0).sum())
sales_negative = int((trans_core["SALES_VALUE"] < 0).sum())
qty_zero = int((trans_core["QUANTITY"] == 0).sum())
qty_negative = int((trans_core["QUANTITY"] < 0).sum())

n_nonpositive_sales = sales_zero + sales_negative
n_nonpositive_qty = qty_zero + qty_negative

print(f"transaction_data.csv — SALES_VALUE <= 0: {n_nonpositive_sales:,} ({n_nonpositive_sales / n_total_trans:.1%}) "
      f"| zero: {sales_zero:,} | negative: {sales_negative:,}")
print(f"transaction_data.csv — QUANTITY <= 0: {n_nonpositive_qty:,} ({n_nonpositive_qty / n_total_trans:.1%}) "
      f"| zero: {qty_zero:,} | negative: {qty_negative:,}")

campaign_dates = pd.read_csv(RAW_DATA_DIR / "campaign_desc.csv", usecols=["CAMPAIGN", "START_DAY", "END_DAY"])
invalid_campaigns = campaign_dates[campaign_dates["END_DAY"] < campaign_dates["START_DAY"]]

print(f"\ncampaign_desc.csv — campaigns with END_DAY < START_DAY: {len(invalid_campaigns)}")
if len(invalid_campaigns) > 0:
    print(invalid_campaigns)

transaction_data.csv — SALES_VALUE <= 0: 18,850 (0.7%) | zero: 18,850 | negative: 0
transaction_data.csv — QUANTITY <= 0: 14,466 (0.6%) | zero: 14,466 | negative: 0

campaign_desc.csv — campaigns with END_DAY < START_DAY: 0


In [21]:
pd.crosstab(
    trans_core["SALES_VALUE"].le(0).rename("SALES_VALUE <= 0"),
    trans_core["QUANTITY"].le(0).rename("QUANTITY <= 0"),
)

QUANTITY <= 0,False,True
SALES_VALUE <= 0,,
False,2576815,67
True,4451,14399


18,850 rows (0.7%) have `SALES_VALUE <= 0` and 14,466 rows (0.6%) have `QUANTITY <= 0`. The first cell splits each count into zeros and negative values, and the cross-tabulation shows how far the two conditions overlap. Both shares are small and the rows are preserved: none are removed, and all 2,595,732 transactions are loaded (Section 7). How these rows are treated in the analysis is decided in `02_rfm.ipynb`.

## 6. Schema design and table creation

Based on the evidence gathered in Sections 3–5 (confirmed dtypes, verified primary keys, and documented duplicate/null handling), we now define the 8 tables in PostgreSQL. Foreign keys are only added where the referenced key is confirmed clean; `household_key → hh_demographic` is intentionally not enforced as a FK, since only 32% of purchasing households have demographic data (Section 4) and a mandatory constraint would break loading for the rest. `causal_data` is created without a primary key: its candidate key is verified in SQL after loading and the constraint is added in Section 8, so that the 36.8M-row load does not pay for index maintenance.

In [22]:
drop_statements = """
DROP TABLE IF EXISTS causal_data CASCADE;
DROP TABLE IF EXISTS coupon_redempt CASCADE;
DROP TABLE IF EXISTS coupon CASCADE;
DROP TABLE IF EXISTS transaction_data CASCADE;
DROP TABLE IF EXISTS campaign_table CASCADE;
DROP TABLE IF EXISTS campaign_desc CASCADE;
DROP TABLE IF EXISTS hh_demographic CASCADE;
DROP TABLE IF EXISTS product CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(drop_statements))

print("Existing tables dropped (if any).")

Existing tables dropped (if any).


The drop step ran without error. Re-running this cell deletes all previously loaded data in these 8 tables, so the loading and key-creation steps (Sections 7 and 8) must be repeated afterwards.

With a clean schema confirmed, we define the 8 tables next.

In [23]:
ddl_statements = """
CREATE TABLE product (
    product_id INTEGER PRIMARY KEY,
    manufacturer INTEGER,
    department VARCHAR(50),
    brand VARCHAR(20),
    commodity_desc VARCHAR(100),
    sub_commodity_desc VARCHAR(100),
    curr_size_of_product VARCHAR(30)
);

CREATE TABLE hh_demographic (
    household_key INTEGER PRIMARY KEY,
    age_desc VARCHAR(20),
    marital_status_code VARCHAR(5),
    income_desc VARCHAR(20),
    homeowner_desc VARCHAR(30),
    hh_comp_desc VARCHAR(30),
    household_size_desc VARCHAR(10),
    kid_category_desc VARCHAR(20)
);

CREATE TABLE campaign_desc (
    campaign INTEGER PRIMARY KEY,
    description VARCHAR(20),
    start_day INTEGER,
    end_day INTEGER
);

CREATE TABLE campaign_table (
    household_key INTEGER NOT NULL,
    campaign INTEGER NOT NULL REFERENCES campaign_desc(campaign),
    description VARCHAR(20),
    PRIMARY KEY (household_key, campaign)
);

CREATE TABLE coupon (
    coupon_upc BIGINT NOT NULL,
    product_id INTEGER NOT NULL REFERENCES product(product_id),
    campaign INTEGER NOT NULL REFERENCES campaign_desc(campaign),
    PRIMARY KEY (coupon_upc, product_id, campaign)
);

CREATE TABLE coupon_redempt (
    household_key INTEGER NOT NULL,
    day INTEGER NOT NULL,
    coupon_upc BIGINT NOT NULL,
    campaign INTEGER NOT NULL REFERENCES campaign_desc(campaign),
    PRIMARY KEY (household_key, coupon_upc, campaign, day)
);

CREATE TABLE transaction_data (
    household_key INTEGER NOT NULL,
    basket_id BIGINT NOT NULL,
    day INTEGER NOT NULL,
    product_id INTEGER NOT NULL REFERENCES product(product_id),
    quantity INTEGER,
    sales_value NUMERIC(10,2),
    store_id INTEGER,
    retail_disc NUMERIC(10,2),
    trans_time INTEGER,
    week_no INTEGER,
    coupon_disc NUMERIC(10,2),
    coupon_match_disc NUMERIC(10,2),
    PRIMARY KEY (household_key, basket_id, product_id)
);

CREATE TABLE causal_data (
    product_id INTEGER NOT NULL REFERENCES product(product_id),
    store_id INTEGER NOT NULL,
    week_no INTEGER NOT NULL,
    display VARCHAR(2),
    mailer VARCHAR(2)
);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_statements))

print("8 tables created successfully.")

8 tables created successfully.


We verify the tables were created as expected by querying PostgreSQL's own system catalog, restricted to the 8 tables of this notebook, rather than assuming the DDL ran correctly. The number of columns in each table is compared with the number of columns in the corresponding CSV.

In [24]:
project_tables = [
    "campaign_desc", "campaign_table", "causal_data", "coupon",
    "coupon_redempt", "hh_demographic", "product", "transaction_data",
]

catalog_query = text("""
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name IN :tables
    ORDER BY table_name, ordinal_position;
""").bindparams(bindparam("tables", expanding=True))

with engine.connect() as conn:
    result = conn.execute(catalog_query, {"tables": project_tables})
    schema_df = pd.DataFrame(result.fetchall(), columns=["table_name", "column_name", "data_type"])

db_columns = schema_df.groupby("table_name").size().rename("db_columns")
column_check = summary_df.set_index("table")[["n_columns"]].join(db_columns)
column_check["match"] = column_check["n_columns"] == column_check["db_columns"]

assert sorted(column_check.index) == sorted(project_tables)
assert column_check["match"].all()
column_check

,n_columns,db_columns,match
table,,,
causal_data,5,5,True
transaction_data,12,12,True
product,7,7,True
coupon,3,3,True
campaign_table,3,3,True
coupon_redempt,4,4,True
hh_demographic,8,8,True
campaign_desc,4,4,True


The catalog query returns exactly the 8 tables created in this notebook, and the number of columns in PostgreSQL equals the number of columns in the CSV for every one of them (the cell fails otherwise). Foreign keys resolved without error, consistent with the creation order (`product` and `campaign_desc` before their dependents).

Before loading, we also check that the declared `VARCHAR` limits are not shorter than the longest value in the raw files, since the schema should be derived from the data and not guessed.

In [25]:
declared_limits = {
    "product.csv": {"DEPARTMENT": 50, "BRAND": 20, "COMMODITY_DESC": 100,
                    "SUB_COMMODITY_DESC": 100, "CURR_SIZE_OF_PRODUCT": 30},
    "hh_demographic.csv": {"AGE_DESC": 20, "MARITAL_STATUS_CODE": 5, "INCOME_DESC": 20,
                           "HOMEOWNER_DESC": 30, "HH_COMP_DESC": 30,
                           "HOUSEHOLD_SIZE_DESC": 10, "KID_CATEGORY_DESC": 20},
    "campaign_desc.csv": {"DESCRIPTION": 20},
    "campaign_table.csv": {"DESCRIPTION": 20},
}

length_rows = []
for filename, limits in declared_limits.items():
    df_len = pd.read_csv(RAW_DATA_DIR / filename, usecols=list(limits), dtype=str)
    for column, declared in limits.items():
        length_rows.append({
            "file": filename,
            "column": column,
            "declared_varchar": declared,
            "max_observed_length": int(df_len[column].str.len().max()),
        })

length_check = pd.DataFrame(length_rows)
assert (length_check["max_observed_length"] <= length_check["declared_varchar"]).all()
length_check

,file,column,declared_varchar,max_observed_length
0,product.csv,DEPARTMENT,50,15
1,product.csv,BRAND,20,8
2,product.csv,COMMODITY_DESC,100,30
3,product.csv,SUB_COMMODITY_DESC,100,30
4,product.csv,CURR_SIZE_OF_PRODUCT,30,10
5,hh_demographic.csv,AGE_DESC,20,5
6,hh_demographic.csv,MARITAL_STATUS_CODE,5,1
7,hh_demographic.csv,INCOME_DESC,20,9
8,hh_demographic.csv,HOMEOWNER_DESC,30,15
9,hh_demographic.csv,HH_COMP_DESC,30,16


Every declared `VARCHAR` limit is at least as large as the longest value observed in the raw files (the cell fails otherwise). The `display` and `mailer` columns of `causal_data` are covered by the load itself: `COPY` raises an error if a value exceeds `VARCHAR(2)`.

Two schema decisions are documented here rather than left implicit:

- `trans_time` is stored as an integer exactly as it appears in the source. The sample values (e.g. `1631`) are consistent with a `HHMM` clock time, but the files do not document the encoding, so it is not converted.
- `household_key` has no foreign key in `transaction_data`, `campaign_table`, or `coupon_redempt`. No table in the source contains the complete list of the 2,500 households: `hh_demographic` is the only candidate parent and covers 32% of them (Section 4), so there is nothing valid to reference.

## 7. Data loading

We load each of the 8 CSVs into its corresponding PostgreSQL table. The six small tables load directly with `pandas.to_sql()`. `transaction_data` and `causal_data`, given their size (Section 3.3), load in chunks through PostgreSQL's `COPY` (Section 7.1). Progress is logged during the chunked loads so a long-running cell is distinguishable from a stalled one. Before loading anything, the notebook checks that the target tables are empty, so re-running this section on already-loaded tables fails with a clear message instead of duplicating rows.

Two tables load from the corrected in-memory versions, not from the raw files, which remain untouched: `coupon` loads from `coupon_clean`, the deduplicated version (Section 5.2), and `product` loads from `product_full`, which has every column found affected in Section 3.1 normalized to `NULL`.

An initial load attempt failed with `UndefinedColumn: column "DESCRIPTION" does not exist` (the PostgreSQL message, shown in Spanish by the local server). The root cause: our `CREATE TABLE` statements use lowercase column names (PostgreSQL convention for unquoted identifiers), but the raw CSVs use uppercase column names in most tables. `pandas.to_sql()` sends column names quoted exactly as they appear in the DataFrame, and PostgreSQL treats quoted identifiers as case-sensitive, so `"DESCRIPTION"` and `description` are seen as two different columns entirely.

The fix is to normalize every DataFrame's column names to lowercase before loading, via a small `to_lowercase_cols()` helper, rather than changing the schema (which is correctly designed following PostgreSQL convention).

In [26]:
def to_lowercase_cols(df):
    return df.rename(columns=str.lower)


def assert_tables_empty(table_names, engine):
    """Fail early if a target table already holds rows (prevents duplicate loads)."""
    with engine.connect() as conn:
        for table_name in table_names:
            n_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
            if n_rows:
                raise RuntimeError(
                    f"Table {table_name} already has {n_rows:,} rows. "
                    "Re-run Section 6 (drop and create) before loading again."
                )


assert_tables_empty(project_tables, engine)

# --- Small tables: direct load ---

small_table_map = {
    "campaign_desc.csv": "campaign_desc",
    "campaign_table.csv": "campaign_table",
    "hh_demographic.csv": "hh_demographic",
}

for filename, table_name in small_table_map.items():
    df = pd.read_csv(RAW_DATA_DIR / filename)
    df = to_lowercase_cols(df)
    df.to_sql(table_name, engine, if_exists="append", index=False)
    print(f"Loaded {filename} -> {table_name} ({len(df)} rows)")

# product: use the already-cleaned in-memory version (whitespace -> NULL)
product_full = to_lowercase_cols(product_full)
product_full.to_sql("product", engine, if_exists="append", index=False)
print(f"Loaded product_full -> product ({len(product_full)} rows)")

# coupon: use the already-deduplicated in-memory version
coupon_clean = to_lowercase_cols(coupon_clean)
coupon_clean.to_sql("coupon", engine, if_exists="append", index=False)
print(f"Loaded coupon_clean -> coupon ({len(coupon_clean)} rows)")

# coupon_redempt: confirmed clean, no transformation needed
coupon_redempt_full = to_lowercase_cols(coupon_redempt_full)
coupon_redempt_full.to_sql("coupon_redempt", engine, if_exists="append", index=False)
print(f"Loaded coupon_redempt_full -> coupon_redempt ({len(coupon_redempt_full)} rows)")

print("\nAll small tables loaded.")

Loaded campaign_desc.csv -> campaign_desc (30 rows)
Loaded campaign_table.csv -> campaign_table (7208 rows)
Loaded hh_demographic.csv -> hh_demographic (801 rows)
Loaded product_full -> product (92353 rows)
Loaded coupon_clean -> coupon (119384 rows)
Loaded coupon_redempt_full -> coupon_redempt (2318 rows)

All small tables loaded.


The six small tables loaded via `to_sql()` without errors. The row counts sent match Section 3 for five of them; `coupon` sends 119,384 rows instead of 124,548 because of the 5,164 exact duplicates removed in Section 5.2. These counts come from the DataFrames; the database counts are checked in Section 7.2.

### 7.1 Fast bulk load for transaction_data and causal_data using COPY

`transaction_data` (2.6M rows) and `causal_data` (36.8M rows) are too large to load row by row via `to_sql()` in reasonable time. We switch to PostgreSQL's native `COPY` command, accessed through psycopg2's `copy_expert`, which streams each chunk in bulk instead of issuing one `INSERT` per row. This is the standard approach for loads of this size. The column list is passed explicitly, so the load does not depend on the column order of the CSV, and each chunk is committed independently and rolled back if it fails.

`causal_data` is read with an explicit dtype: without it, pandas raises a `DtypeWarning` on `display` because it infers different types in different chunks. Reading `display` and `mailer` as text keeps every code exactly as it appears in the file and matches the `VARCHAR` columns. For `transaction_data`, the loop also accumulates the quantity and sales totals of the CSV, which Section 7.2 compares against the database.

In [27]:
def copy_df_to_table(df, table_name, columns, engine):
    buffer = io.StringIO()
    df[columns].to_csv(buffer, index=False, header=False, sep="\t", na_rep="\\N")
    buffer.seek(0)
    raw_conn = engine.raw_connection()
    cursor = raw_conn.cursor()
    try:
        cursor.copy_expert(
            f"COPY {table_name} ({', '.join(columns)}) FROM STDIN WITH (FORMAT text, NULL '\\N')",
            buffer,
        )
        raw_conn.commit()
    except Exception:
        raw_conn.rollback()
        raise
    finally:
        cursor.close()
        raw_conn.close()


# --- transaction_data ---

transaction_columns = [
    "household_key", "basket_id", "day", "product_id", "quantity", "sales_value",
    "store_id", "retail_disc", "trans_time", "week_no", "coupon_disc", "coupon_match_disc",
]
transaction_totals = {"rows": 0, "quantity": 0, "sales_value": 0.0}
chunk_size = 500_000
log_every = 5

print("--- Loading transaction_data.csv -> transaction_data (via COPY) ---")
for i, chunk in enumerate(pd.read_csv(RAW_DATA_DIR / "transaction_data.csv", chunksize=chunk_size), start=1):
    chunk = to_lowercase_cols(chunk)
    copy_df_to_table(chunk, "transaction_data", transaction_columns, engine)
    transaction_totals["rows"] += len(chunk)
    transaction_totals["quantity"] += int(chunk["quantity"].sum())
    transaction_totals["sales_value"] += float(chunk["sales_value"].sum())
    if i % log_every == 0:
        print(f"  chunk {i} — {transaction_totals['rows']:,} rows loaded so far")
print(f"Finished transaction_data.csv: {transaction_totals['rows']:,} total rows loaded.\n")

# --- causal_data ---

causal_columns = ["product_id", "store_id", "week_no", "display", "mailer"]
causal_dtypes = {"PRODUCT_ID": "int64", "STORE_ID": "int64", "WEEK_NO": "int64",
                 "display": "str", "mailer": "str"}
chunk_size = 2_000_000
causal_rows = 0

print("--- Loading causal_data.csv -> causal_data (via COPY) ---")
for i, chunk in enumerate(pd.read_csv(RAW_DATA_DIR / "causal_data.csv",
                                      dtype=causal_dtypes, chunksize=chunk_size), start=1):
    chunk = to_lowercase_cols(chunk)
    copy_df_to_table(chunk, "causal_data", causal_columns, engine)
    causal_rows += len(chunk)
    if i % log_every == 0:
        print(f"  chunk {i} — {causal_rows:,} rows loaded so far")
print(f"Finished causal_data.csv: {causal_rows:,} total rows loaded.")

assert transaction_totals["rows"] == file_summaries["transaction_data.csv"]["n_rows"]
assert causal_rows == file_summaries["causal_data.csv"]["n_rows"]

--- Loading transaction_data.csv -> transaction_data (via COPY) ---
  chunk 5 — 2,500,000 rows loaded so far
Finished transaction_data.csv: 2,595,732 total rows loaded.

--- Loading causal_data.csv -> causal_data (via COPY) ---
  chunk 5 — 10,000,000 rows loaded so far
  chunk 10 — 20,000,000 rows loaded so far
  chunk 15 — 30,000,000 rows loaded so far
Finished causal_data.csv: 36,786,524 total rows loaded.


Both loads finished without errors, and the rows sent to PostgreSQL match the row counts confirmed in Section 3.3 (the cell asserts it). No `DtypeWarning` is raised, because the promotional columns are read with an explicit dtype. What the database actually holds is verified next.

### 7.2 Verifying the load in PostgreSQL

Counting the rows that were sent is not the same as checking what PostgreSQL stored. We therefore query the database itself: row counts per table against the expected counts, the `NULL` count of every `product` column normalized in Section 3.1 (not just `curr_size_of_product`), quantity and sales totals of `transaction_data` against the CSV totals accumulated during the load, and the codes stored in `causal_data.display` and `causal_data.mailer` (no `NULL` and no literal `'nan'` should have been written by the load). The cell also checks that `campaign_table.description` never disagrees with `campaign_desc.description`, since `campaign_table` repeats that attribute.

In [28]:
expected = {name.replace(".csv", ""): data["n_rows"] for name, data in file_summaries.items()}
expected["coupon"] = len(coupon_clean)  # 124,548 minus the exact duplicates removed in Section 5.2

count_rows = []
with engine.connect() as conn:
    for table_name, n_expected in expected.items():
        n_db = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
        count_rows.append({"table": table_name, "expected": n_expected, "in_database": n_db})

    n_null_by_column = {}
    for column in normalized_counts:
        n_null_by_column[column] = conn.execute(text(
            f"SELECT COUNT(*) FROM product WHERE {column.lower()} IS NULL"
        )).scalar()

    db_sales, db_qty = conn.execute(text(
        "SELECT SUM(sales_value), SUM(quantity) FROM transaction_data"
    )).one()

    n_bad = {}
    for col in ("display", "mailer"):
        n_bad[col] = conn.execute(text(
            f"SELECT COUNT(*) FROM causal_data WHERE {col} IS NULL OR {col} = 'nan'"
        )).scalar()

    display_values = pd.read_sql(text(
        "SELECT display AS code, COUNT(*) AS n_rows FROM causal_data GROUP BY display ORDER BY display"
    ), conn)
    mailer_values = pd.read_sql(text(
        "SELECT mailer AS code, COUNT(*) AS n_rows FROM causal_data GROUP BY mailer ORDER BY mailer"
    ), conn)

    n_desc_mismatch = conn.execute(text("""
        SELECT COUNT(*)
        FROM campaign_table ct
        JOIN campaign_desc cd USING (campaign)
        WHERE ct.description IS DISTINCT FROM cd.description
    """)).scalar()

count_check = pd.DataFrame(count_rows)
count_check["match"] = count_check["expected"] == count_check["in_database"]
print(count_check.to_string(index=False))

print()
for column, n_db in n_null_by_column.items():
    print(f"product.{column.lower()} NULLs: {n_db:,} (Section 3.1: {normalized_counts[column]:,})")
print(f"\ntransaction_data SUM(quantity):    DB {int(db_qty):,} | CSV {transaction_totals['quantity']:,}")
print(f"transaction_data SUM(sales_value): DB {float(db_sales):,.2f} | CSV {transaction_totals['sales_value']:,.2f}")
print(f"\ncausal_data rows with NULL or 'nan': display = {n_bad['display']:,}, mailer = {n_bad['mailer']:,}")
print("\ndisplay codes stored:")
print(display_values.to_string(index=False))
print("\nmailer codes stored:")
print(mailer_values.to_string(index=False))
print(f"\ncampaign_table rows whose description differs from campaign_desc: {n_desc_mismatch:,}")

assert count_check["match"].all()
for column, n_db in n_null_by_column.items():
    assert n_db == normalized_counts[column]
assert int(db_qty) == transaction_totals["quantity"]
assert math.isclose(float(db_sales), transaction_totals["sales_value"], abs_tol=1.0)
assert n_bad["display"] == 0 and n_bad["mailer"] == 0
assert n_desc_mismatch == 0

           table  expected  in_database  match
   campaign_desc        30           30   True
  campaign_table      7208         7208   True
     causal_data  36786524     36786524   True
          coupon    119384       119384   True
  coupon_redempt      2318         2318   True
  hh_demographic       801          801   True
         product     92353        92353   True
transaction_data   2595732      2595732   True

product.commodity_desc NULLs: 15 (Section 3.1: 15)
product.curr_size_of_product NULLs: 30,607 (Section 3.1: 30,607)
product.department NULLs: 15 (Section 3.1: 15)
product.sub_commodity_desc NULLs: 15 (Section 3.1: 15)

transaction_data SUM(quantity):    DB 260,685,622 | CSV 260,685,622
transaction_data SUM(sales_value): DB 8,057,463.08 | CSV 8,057,463.08

causal_data rows with NULL or 'nan': display = 0, mailer = 0

display codes stored:
code   n_rows
   0 21038745
   1  1102141
   2  1812840
   3  2073738
   4   592985
   5  2575289
   6  1816021
   7  2362118
   9  26

All 8 tables match the expected row counts: `causal_data` has 36,786,524 rows, as in Section 3, and `coupon` has 119,384 rows (124,548 minus the 5,164 exact duplicates removed in Section 5.2). Every `product` column normalized in Section 3.1 holds the same `NULL` count in PostgreSQL as it did in memory (the cell asserts it column by column), and the quantity and sales totals of `transaction_data` match the CSV, so no rows or values were altered on the way in. No `NULL` or literal `'nan'` reached `display` or `mailer`; the code inventories above document exactly which promotional codes are stored.

Finally, `campaign_table.description` is a redundant copy of `campaign_desc.description`: the two never disagree. It is kept to mirror the source file, and `campaign_desc` is the reference for campaign attributes in the notebooks that follow.

## 8. Verifying and adding the primary key of `causal_data`

`causal_data` was created without a primary key (Section 6), and it was left out of the duplicate check in Section 5 because of its size. Now that it is loaded, the checks run inside PostgreSQL: the database groups the 36.8M rows itself, without holding tens of millions of tuples in Python memory, and it compares the values exactly as stored, so pandas type inference plays no role.

### 8.1 Is (`product_id`, `store_id`, `week_no`) unique?

The candidate key is the natural grain of promotional data: one row per product, store, and week. We count how many keys appear more than once and how many rows exceed the first occurrence of each key.

In [29]:
with engine.connect() as conn:
    n_causal_total = conn.execute(text("SELECT COUNT(*) FROM causal_data")).scalar()
    n_repeated_keys, n_extra_rows = conn.execute(text("""
        SELECT COUNT(*) AS repeated_keys, COALESCE(SUM(n - 1), 0) AS extra_rows
        FROM (
            SELECT product_id, store_id, week_no, COUNT(*) AS n
            FROM causal_data
            GROUP BY product_id, store_id, week_no
            HAVING COUNT(*) > 1
        ) t
    """)).one()

n_extra_rows = int(n_extra_rows)
print(f"Total rows: {n_causal_total:,}")
print(f"Distinct (product_id, store_id, week_no) combinations: {n_causal_total - n_extra_rows:,}")
print(f"Keys that appear more than once: {n_repeated_keys:,}")
print(f"Duplicate rows on this key: {n_extra_rows:,} ({n_extra_rows / n_causal_total:.3%})")

Total rows: 36,786,524
Distinct (product_id, store_id, week_no) combinations: 36,771,279
Keys that appear more than once: 15,245
Duplicate rows on this key: 15,245 (0.041%)


The candidate key (`product_id`, `store_id`, `week_no`) is not unique: 15,245 duplicate rows out of 36,786,524 (0.041%). Before treating this as a data quality problem, we apply the same logic used for `coupon.csv` in Section 5.1: these three columns are not the entire row. `display` and `mailer` are missing from the check, so a "duplicate" key could simply mean the same product/store/week had two distinct promotional events logged separately (e.g. one display promotion and one mailer promotion). We re-check using all 5 columns to tell the two scenarios apart.

### 8.2 Checking for fully duplicated rows

In [30]:
with engine.connect() as conn:
    n_full_duplicates = conn.execute(text("""
        SELECT COALESCE(SUM(n - 1), 0)
        FROM (
            SELECT COUNT(*) AS n
            FROM causal_data
            GROUP BY product_id, store_id, week_no, display, mailer
            HAVING COUNT(*) > 1
        ) t
    """)).scalar()

print(f"Fully duplicated rows (all 5 columns): {int(n_full_duplicates):,}")

Fully duplicated rows (all 5 columns): 0


No row in `causal_data` is fully duplicated: the 15,245 repeats on (`product_id`, `store_id`, `week_no`) differ in `display` and/or `mailer`. This rules out exact duplicate rows, but it does not show that they are valid separate promotional events; they could also be conflicting records for the same key. They are kept unmodified.

The primary key is therefore defined over all 5 columns as a technical constraint. Consequently, (`product_id`, `store_id`, `week_no`) is not a unique grain: joining `causal_data` with `transaction_data` on those keys can multiply rows for those 15,245 keys. Their interpretation is left to the notebook that uses promotional context.

### 8.3 Adding the primary key

We add the primary key with `ALTER TABLE`, after the load, so the 36.8M rows were not indexed while being inserted. Building the index can take several minutes.

In [31]:
alter_statement = """
ALTER TABLE causal_data DROP CONSTRAINT IF EXISTS causal_data_pkey;
ALTER TABLE causal_data ADD PRIMARY KEY (product_id, store_id, week_no, display, mailer);
"""

with engine.begin() as conn:
    conn.execute(text(alter_statement))

print("Primary key added to causal_data.")

Primary key added to causal_data.


The primary key was added successfully with no conflict, consistent with the check in Section 8.2. Note that a 5-column key means `causal_data` has no natural key at the (`product_id`, `store_id`, `week_no`) grain. The schema design phase is now complete: all 8 tables exist in PostgreSQL with primary keys grounded in the checks above, and foreign keys wired wherever the referenced key was confirmed clean.

### 8.4 Updating planner statistics

`ANALYZE` refreshes PostgreSQL's table statistics, so the queries in the following notebooks start with accurate row estimates instead of the empty statistics of freshly loaded tables.

In [32]:
with engine.begin() as conn:
    for table_name in project_tables:
        conn.execute(text(f"ANALYZE {table_name};"))

print("Planner statistics updated for all 8 tables.")

Planner statistics updated for all 8 tables.


## 9. Relational structure overview

With all 8 tables loaded and their primary/foreign keys enforced by
PostgreSQL, we document how they relate to each other, queried directly
from the database's own constraint metadata, rather than hand-typed, so this
stays accurate if the schema ever changes.

In [33]:
fk_query = text("""
    SELECT
        tc.table_name AS child_table,
        kcu.column_name AS child_column,
        ccu.table_name AS parent_table,
        ccu.column_name AS parent_column
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema = kcu.table_schema
    JOIN information_schema.constraint_column_usage ccu
        ON tc.constraint_name = ccu.constraint_name
       AND tc.table_schema = ccu.table_schema
    WHERE tc.constraint_type = 'FOREIGN KEY'
      AND tc.table_schema = 'public'
      AND tc.table_name IN :tables
    ORDER BY child_table, child_column;
""").bindparams(bindparam("tables", expanding=True))

with engine.connect() as conn:
    result = conn.execute(fk_query, {"tables": project_tables})
    fk_df = pd.DataFrame(result.fetchall(), columns=["child_table", "child_column", "parent_table", "parent_column"])

fk_df

,child_table,child_column,parent_table,parent_column
0,campaign_table,campaign,campaign_desc,campaign
1,causal_data,product_id,product,product_id
2,coupon,campaign,campaign_desc,campaign
3,coupon,product_id,product,product_id
4,coupon_redempt,campaign,campaign_desc,campaign
5,transaction_data,product_id,product,product_id


The output confirms every foreign key defined in Section 6: `campaign_table`,
`coupon`, and `coupon_redempt` reference `campaign_desc`; `coupon`,
`transaction_data`, and `causal_data` reference `product`. One relationship
is deliberately absent from this list: `household_key` between
`transaction_data` and `hh_demographic`. As established in Section 4, only
32% of purchasing households have demographic data, so this relationship is
documented here in prose rather than enforced as a database constraint: a
mandatory FK would have made loading fail for the other 68%.

More generally, no table in the source contains the complete list of the 2,500 households, so `household_key` in `transaction_data`, `campaign_table`, and `coupon_redempt` cannot reference a household dimension: `hh_demographic` is the only candidate parent and covers 32% of them.

## 10. Conclusion

This notebook set out to answer one question: is this dataset consistent and complete enough to support a descriptive analysis of customer behavior? The answer is yes, within the limitations below.

Two corrections were applied before loading and documented: 5,164 exact duplicate rows removed from `coupon.csv` (4.1%) and whitespace-only values converted to `NULL` in every affected column of `product.csv`, found by scanning all text columns of the six small tables rather than assuming the 5-row sample's single flagged column (`CURR_SIZE_OF_PRODUCT`) was the only one; the exact columns and counts are reported in Section 3.1. The raw files are untouched. One additional finding was preserved without modification: 15,245 repeated (`product_id`, `store_id`, `week_no`) keys in `causal_data` (0.041%), which differ in `display` and/or `mailer` and whose nature is not established here (Section 8.1).

Demographic coverage is partial: only 32.0% of purchasing households (801 of 2,500) have a record in `hh_demographic` (Section 4). Missing demographics are not imputed, and this scope will be stated wherever demographic segmentation is used.

Temporal coverage is consistent in units (`DAY`, `WEEK_NO`; no calendar dates), with two points to carry forward (Section 3.4). First, `causal_data` covers weeks 9 to 101, leaving weeks 1–8 and week 102 of the transactions without promotional context; the final week may also contain fewer than seven days of transactions. Second, `campaign_desc` records `END_DAY` values up to 719, beyond the last transaction day (711), so campaign-window metrics must handle campaigns that end after the observation window.

Non-positive values are preserved: 0.7% of transactions have `SALES_VALUE <= 0` and 0.6% have `QUANTITY <= 0`. Their treatment is decided in `02_rfm.ipynb` (Section 5.3).

The database is loaded and verified against the source files (row counts, `NULL` counts, and quantity and sales totals; Section 7.2), and its relational structure is documented (Section 9). `01_data_setup_and_loading.ipynb` is complete; `02_rfm.ipynb` can build on it.